# Explore Event Precipitation and Meteorology

In [1]:
import os
os.chdir('/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo/')

# general
import glob
import datetime as dt
from pathlib import Path

# data 
import xarray as xr 
import numpy as np
import pandas as pd

# plotting
import matplotlib.pyplot as plt
import plotly.express as px 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Configure Plotly for Jupyter notebooks
pio.renderers.default = "notebook"
# Alternative renderers you can try if "notebook" doesn't work:
# pio.renderers.default = "plotly_mimetype+notebook"
# pio.renderers.default = "jupyter_lab"

# helper tools
from metpy import calc, units
import scipy.stats as stats
from sklearn.linear_model import LinearRegression

In [2]:
colors = ["#D81B60", "#1E88E5", "#004D40", "#FFC107", "#281733"]

In [3]:
def get_file_destination(DATA_DIR, SRC, PRODUCT, WITH_MET, RAW_OR_NORMALIZED):
    if WITH_MET and RAW_OR_NORMALIZED == "raw":
        WITH_MET = "_with_raw_met"
    elif WITH_MET and RAW_OR_NORMALIZED == "normalized":
        WITH_MET = "_with_normalized_met"
    else:
        WITH_MET = ""

    if PRODUCT == "gridded":
        PRODUCT_NAME = "_gridded"
        FOLDER_NAME = "gridded_events"
    elif PRODUCT == "events":
        PRODUCT_NAME = ""
        FOLDER_NAME = "events"
    else:
        PRODUCT_NAME = ""
    if SRC in ['asfs', 'sos']:
        SITE = 'kettle_ponds'
    elif SRC in ['bb', 'sail']:
        SITE = 'gothic'
    else:
        SITE = input("Enter site name (gothic or kettle_ponds): ")
    return DATA_DIR / SITE / f"{FOLDER_NAME}{WITH_MET}" / f"{SITE}{PRODUCT_NAME}_precipitation_event_comparisons_{SRC}{WITH_MET}.nc"

In [4]:
DATA_DIR = '/storage/dlhogan/precipitation-rodeo/data/for_analysis'
# Define your root and sites
DATA_DIR = Path(DATA_DIR)
STORAGE_DIR = '/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo/04_products/figures/events'
SRC = "asfs" # one of [bb, sail, asfs, sos, '']
PRODUCT = "events" # or events
WITH_MET = True  # or True
RAW_OR_NORMALIZED = "raw"  # or raw
file_dest = get_file_destination(DATA_DIR, SRC, PRODUCT, WITH_MET, RAW_OR_NORMALIZED)
print(f"Loading data from: {file_dest}")
ds = xr.open_dataset(file_dest)

Loading data from: /storage/dlhogan/precipitation-rodeo/data/for_analysis/kettle_ponds/events_with_raw_met/kettle_ponds_precipitation_event_comparisons_asfs_with_raw_met.nc


In [5]:
gothic_ppt_ds = xr.open_dataset('/storage/dlhogan/precipitation-rodeo/data/processed/final/gothic_precipitation_30min.nc')
gothic_met_ds = xr.open_dataset('/storage/dlhogan/precipitation-rodeo/data/processed/SAIL/met_30min.nc')
kettle_ponds_ppt_ds = xr.open_dataset('/storage/dlhogan/precipitation-rodeo/data/processed/final/kettle_ponds_precipitation_30min_with_flags.nc')
kettle_ponds_met_ds = xr.open_dataset('/storage/dlhogan/precipitation-rodeo/data/processed/SPLASH/asfs30_30min.nc')
billy_met_ds = xr.open_dataset('/storage/dlhogan/precipitation-rodeo/data/processed/billy_barr/billy_barr_20211001-20230930_30min.nc')
sos_vars = ['RH_3m_c', 'T_3m_c', 'P_10m_c', 'u_10m_c', 'v_10m_c','SF_avg_1m_ue', 'SF_avg_2m_ue']
sos_ds = xr.open_dataset('/storage/dlhogan/precipitation-rodeo/data/processed/SOS/sos_ds_30min.nc')[sos_vars]

# calcualte wind direction
u = gothic_met_ds['u'].metpy.convert_units('m/s')
v = gothic_met_ds['v'].metpy.convert_units('m/s')
wind_dir = calc.wind_direction(u, v).metpy.dequantify()
gothic_met_ds['wind_dir'] = wind_dir

splash_variable_rename_map = {
    'atmos_pressure': 'atmos_pressure',
    'rh': 'rh_mean',
    'temp': 'temp_mean',
    'wspd_u_mean': 'u',
    'wspd_v_mean': 'v',
}

kettle_ponds_met_ds = kettle_ponds_met_ds.rename(splash_variable_rename_map)
# append a new variable to the dataset, wspd_vec_mean
kettle_ponds_met_ds = kettle_ponds_met_ds.assign(wspd_vec_mean=np.sqrt(kettle_ponds_met_ds['u']**2 + kettle_ponds_met_ds['v']**2))

sos_variable_rename_map = {
    'P_10m_c': 'atmos_pressure',
    'RH_3m_c': 'rh_mean',
    'T_3m_c': 'temp_mean',
    'u_10m_c': 'u',
    'v_10m_c': 'v',
}

sos_ds = sos_ds.rename(sos_variable_rename_map)

# append a new variable to the dataset, wspd_vec_mean
sos_ds = sos_ds.assign(wspd_vec_mean=np.sqrt(sos_ds['u']**2 + sos_ds['v']**2))

bb_met_rename_map = {
    'baromPress': 'atmos_pressure',
    'relHumidty': 'rh_mean',
    'avAirTemp': 'temp_mean',
    'windDirec': 'wind_dir',
    'windSpeed': 'wspd_vec_mean',
}

billy_met_ds = billy_met_ds.rename(bb_met_rename_map)

# Measured Events

In [6]:
# set benchmark site 
SITE = 'kettle_ponds'  # one of [gothic, kettle_ponds]
BENCHMARK = 'billy_barr_precip'
EVENTS = slice(1,11) # top 10 events

INSTRUMENT = "splash_pluvio"
OUTPUT_DIR = os.path.join(STORAGE_DIR, f"{SITE}/{INSTRUMENT}")

ppt_ds = kettle_ponds_ppt_ds
met_ds = kettle_ponds_met_ds
# if output dir does not exist
os.makedirs(OUTPUT_DIR+f'/bb_met', exist_ok=True)

print(list(ppt_ds.data_vars))

['billy_barr_precip', 'qc_bad_billy_barr_precip', 'qc_missing_billy_barr_precip', 'splash_pluvio', 'qc_missing_splash_pluvio', 'qc_bad_splash_pluvio', 'splash_ld_unadjusted', 'qc_missing_splash_ld_unadjusted', 'qc_bad_splash_ld_unadjusted', 'splash_ld_holyroyd', 'splash_ld_brandes', 'splash_ld_heymsfield', 'precip_type', 'sos_SWE_p1', 'qc_SWE_p1_missing', 'qc_SWE_p1_bad', 'sos_SWE_p2', 'qc_SWE_p2_missing', 'qc_SWE_p2_bad', 'sos_SWE_p3', 'qc_SWE_p3_missing', 'qc_SWE_p3_bad', 'sos_SWE_p4', 'qc_SWE_p4_missing', 'qc_SWE_p4_bad', 'sail_squire_m2009_1', 'sail_squire_m2009_2', 'sail_squire_ws88diw', 'sail_squire_ws2012', 'qc_missing_sail_squire_m2009_1', 'qc_bad_sail_squire_m2009_1']


In [7]:
for event in range(1,11):
    EVENT = event  # change to explore different events
    event_ds = ds.sel(event_id=EVENT, test_instrument=INSTRUMENT, benchmark=BENCHMARK)

    start, end = pd.to_datetime(event_ds['start_time'].values), pd.to_datetime(event_ds['end_time'].values)
    event_ppt = ppt_ds.sel(time=slice(start, end))
    event_met = met_ds.sel(time=slice(start, end))
    event_bb_met = billy_met_ds.sel(time=slice(start, end))
    # calcualte wind direction
    u = event_met['u'].metpy.convert_units('m/s')
    v = event_met['v'].metpy.convert_units('m/s')
    wind_dir = calc.wind_direction(u, v).metpy.dequantify()
    event_met['wind_dir'] = wind_dir
    event_ppt_rate = (event_ppt*2)#.rolling(time=2, center=True).mean()  # mm per 30 min to mm per hour
    print(event_ds['start_time'].values, event_ds['end_time'].values)

    # make a plotly plot of the event with ppt and met data
    fig = make_subplots(rows=5, cols=1, 
                        shared_xaxes=True, 
                        subplot_titles=("Precipitation", "Air Temperature", "Wind Speed", "Relative Humidity", "Pressure"),
                        vertical_spacing=0.05,
                        specs=[[{"secondary_y": True}],  # only first subplot
                                [{}], 
                                [{"secondary_y": True}],
                                [{}],
                                [{}]])
    # Cumulative precipitation
    fig.add_trace(go.Scatter(x=event_ppt['time'].values, 
                            y=event_ppt[BENCHMARK].cumsum().values, 
                            name=f'{BENCHMARK.replace("_", " ")} (mm)',
                            mode='lines',
                            line=dict(width=4, color=colors[0])), 
                            row=1, col=1, )
    fig.add_trace(go.Scatter(x=event_ppt['time'].values, 
                            y=event_ppt[INSTRUMENT].cumsum().values, 
                            name=f'{INSTRUMENT.replace("_", " ")} (mm)',
                            mode='lines',
                            line=dict(width=4, color=colors[1])), 
                            row=1, col=1, )
    # Precipitation rate
    # secondary y-axis for precipitation rate
    fig.add_trace(go.Scatter(x=event_ppt_rate['time'].values, 
                            y=event_ppt_rate[BENCHMARK].values, 
                            name=f'{BENCHMARK.replace("_", " ")} Rate (mm/hr)',
                            mode='lines',
                            line=dict(width=2, dash='dot', color=colors[0]),
                            yaxis='y2'),
                            secondary_y=True,
                            row=1, col=1, )
    fig.add_trace(go.Scatter(x=event_ppt_rate['time'].values, 
                            y=event_ppt_rate[INSTRUMENT].values, 
                            name=f'{INSTRUMENT.replace("_", " ")} Rate (mm/hr)',
                            mode='lines',
                            line=dict(width=2, dash='dot', color=colors[1]),
                            yaxis='y2'),
                            secondary_y=True,
                            row=1, col=1, )
    # Temperature
    fig.add_trace(go.Scatter(x=event_met['time'].values, 
                            y=event_met['temp_mean'].values, 
                            mode='lines', 
                            name='Air Temperature (°C)',
                            line=dict(width=4,)),
                            row=2, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=5, row=2, col=1)
    # Wind speed
    fig.add_trace(go.Scatter(x=event_met['time'].values, 
                            y=event_met['wspd_vec_mean'].values, 
                            mode='lines', 
                            name='Wind Speed (m/s)',
                            line=dict(width=4,)), 
                            row=3, col=1)
    # Wind direction as secondary y-axis
    fig.add_trace(go.Scatter(x=event_met['time'].values, 
                            y=event_met['wind_dir'].values, 
                            mode='markers', 
                            name='Wind Direction (°)',
                            line=dict(width=2, dash='dot', color='gray')),
                            secondary_y=True,
                            row=3, col=1)
    fig.add_hline(y=180, line_dash="dash", line_color="black", line_width=5, row=3, col=1, secondary_y=True)
    # Plot relative humidity on 4th subplot
    fig.add_trace(go.Scatter(x=event_met['time'].values, 
                            y=event_met['rh_mean'].values, 
                            mode='lines', 
                            name='Relative Humidity (%)',
                            line=dict(width=4, color=colors[2])),
                            row=4, col=1)
    # Plot pressure on 5th subplot
    fig.add_trace(go.Scatter(x=event_met['time'].values,
                                y=event_met['atmos_pressure'].values*10,
                                mode='lines',
                                name='Pressure (hPa)',
                                line=dict(width=4, color='magenta')),
                                row=5, col=1)

    fig.update_layout( title_text=f'Event ID: {event_ds["event_id"].values} from {event_ds["start_time"].dt.date.values} to {event_ds["end_time"].dt.date.values}')

    # update yaxis titles
    fig.update_yaxes(title_text="Cumulative<br>Precipitation (mm)", row=1, col=1)
    fig.update_layout(yaxis2=dict(title='Precipitation Rate<br>(mm/hr)', overlaying='y', side='right', position=0.15))
    fig.update_yaxes(title_text="Wind Speed (m/s)", row=3, col=1)
    fig.update_yaxes(title_text="Wind Direction (°)", secondary_y=True, row=3, col=1)
    fig.update_yaxes(title_text="Air<br>Temperature (°C)", row=2, col=1)


    # Update layout to enable vertical line (spike) on all subplots
    for axis in ['xaxis', 'xaxis2', 'xaxis3']:
        fig.layout[axis].update(
            showspikes=True,
            spikemode='across',
            spikesnap='cursor',
            spikecolor='gray',
            spikethickness=1
        )
    # move the legend to the bottom with 4 columns
    fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5, traceorder="normal", font=dict(size=18)), 
                    height=1000, width=800,
                    hovermode='x',  # vertical line and combined tooltip
                    )
    fig.update_yaxes(showgrid=False, secondary_y=True, row=1, col=1)
    fig.update_yaxes(showgrid=False, secondary_y=True, row=3, col=1)
    # decrease vertical white space between each subplot
    fig.update_layout(margin=dict(t=100, b=100, l=50, r=50), 
                    title_font_size=24)
    fig.update_xaxes(range=[pd.to_datetime(event_bb_met['time'].min().values), pd.to_datetime(event_bb_met['time'].max().values)])
    fig.write_image(f'{OUTPUT_DIR}/{SITE}_event_{event_ds["event_id"].values}_exploration_{INSTRUMENT}_vs_{BENCHMARK}.png', scale=2,
                    height=1000, width=1000)
    
        # make a plotly plot of the event with ppt and met data
    fig = make_subplots(rows=5, cols=1, 
                        shared_xaxes=True, 
                        subplot_titles=("Precipitation", "Air Temperature", "Wind Speed", "Relative Humidity"),
                        vertical_spacing=0.05,
                        specs=[[{"secondary_y": True}],  # only first subplot
                                [{}], 
                                [{"secondary_y": True}],
                                [{}],
                                [{}]])
    # Cumulative precipitation
    fig.add_trace(go.Scatter(x=event_ppt['time'].values, 
                            y=event_ppt[BENCHMARK].cumsum().values, 
                            name=f'{BENCHMARK.replace("_", " ")} (mm)',
                            mode='lines',
                            line=dict(width=4, color=colors[0])), 
                            row=1, col=1, )
    fig.add_trace(go.Scatter(x=event_ppt['time'].values, 
                            y=event_ppt[INSTRUMENT].cumsum().values, 
                            name=f'{INSTRUMENT.replace("_", " ")} (mm)',
                            mode='lines',
                            line=dict(width=4, color=colors[1])), 
                            row=1, col=1, )
    # Precipitation rate
    # secondary y-axis for precipitation rate
    fig.add_trace(go.Scatter(x=event_ppt_rate['time'].values, 
                            y=event_ppt_rate[BENCHMARK].values, 
                            name=f'{BENCHMARK.replace("_", " ")} Rate (mm/hr)',
                            mode='lines',
                            line=dict(width=2, dash='dot', color=colors[0]),
                            yaxis='y2'),
                            secondary_y=True,
                            row=1, col=1, )
    fig.add_trace(go.Scatter(x=event_ppt_rate['time'].values, 
                            y=event_ppt_rate[INSTRUMENT].values, 
                            name=f'{INSTRUMENT.replace("_", " ")} Rate (mm/hr)',
                            mode='lines',
                            line=dict(width=2, dash='dot', color=colors[1]),
                            yaxis='y2'),
                            secondary_y=True,
                            row=1, col=1, )
    # Temperature
    fig.add_trace(go.Scatter(x=event_bb_met['time'].values, 
                            y=event_bb_met['temp_mean'].values, 
                            mode='lines', 
                            name='Air Temperature (°C)',
                            line=dict(width=4,)),
                            row=2, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=5, row=2, col=1)
    # Wind speed
    fig.add_trace(go.Scatter(x=event_bb_met['time'].values, 
                            y=event_bb_met['wspd_vec_mean'].values, 
                            mode='lines', 
                            name='Wind Speed (m/s)',
                            line=dict(width=4,)), 
                            row=3, col=1)
    # Wind direction as secondary y-axis
    fig.add_trace(go.Scatter(x=event_bb_met['time'].values, 
                            y=event_bb_met['wind_dir'].values, 
                            mode='markers', 
                            name='Wind Direction (°)',
                            line=dict(width=2, dash='dot', color='gray')),
                            secondary_y=True,
                            row=3, col=1)
    fig.add_hline(y=180, line_dash="dash", line_color="black", line_width=5, row=3, col=1, secondary_y=True)
    # Plot relative humidity on 4th subplot
    fig.add_trace(go.Scatter(x=event_bb_met['time'].values, 
                            y=event_bb_met['rh_mean'].values, 
                            mode='lines', 
                            name='Relative Humidity (%)',
                            line=dict(width=4, color=colors[2])),
                            row=4, col=1)
    # Plot barometric pressure on 5th subplot
    fig.add_trace(go.Scatter(x=event_bb_met['time'].values, 
                            y=event_bb_met['atmos_pressure'].values, 
                            mode='lines', 
                            name='Pressure (hPa)',
                            line=dict(width=4, color='magenta')),
                            row=5, col=1)

    fig.update_layout( title_text=f'Event ID: {event_ds["event_id"].values} from {event_ds["start_time"].dt.date.values} to {event_ds["end_time"].dt.date.values}')
    fig.layout['yaxis3'].update(showgrid=False)
    # update yaxis titles
    fig.update_yaxes(title_text="Cumulative<br>Precipitation (mm)", row=1, col=1)
    fig.update_layout(yaxis2=dict(title='Precipitation Rate<br>(mm/hr)', overlaying='y', side='right', position=0.15))
    fig.update_yaxes(title_text="Wind Speed (m/s)", row=3, col=1)
    fig.update_yaxes(title_text="Wind Direction (°)", secondary_y=True, row=3, col=1)
    fig.update_yaxes(title_text="Air Temperature (°C)", row=2, col=1)


    # Update layout to enable vertical line (spike) on all subplots
    for axis in ['xaxis', 'xaxis2', 'xaxis3']:
        fig.layout[axis].update(
            showspikes=True,
            spikemode='across',
            spikesnap='cursor',
            spikecolor='gray',
            spikethickness=1
        )
    # move the legend to the bottom with 4 columns
    fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5, traceorder="normal", font=dict(size=18)), 
                    height=1000, width=800,
                    hovermode='x',  # vertical line and combined tooltip
                    )
    fig.update_yaxes(showgrid=False, secondary_y=True, row=1, col=1)
    fig.update_yaxes(showgrid=False, secondary_y=True, row=3, col=1)
    # decrease vertical white space between each subplot
    fig.update_layout(margin=dict(t=100, b=100, l=50, r=50), 
                    title_font_size=24)
    # update x-axis to limits of the event
    fig.update_xaxes(range=[pd.to_datetime(event_bb_met['time'].min().values), pd.to_datetime(event_bb_met['time'].max().values)])
    fig.write_image(f'{OUTPUT_DIR}/bb_met/{SITE}_event_{event_ds["event_id"].values}_exploration_{INSTRUMENT}_vs_{BENCHMARK}.png', scale=2,
    height=1000, width=1000)

2023-03-30T10:00:00.000000000 2023-04-01T17:30:00.000000000
2022-04-16T07:00:00.000000000 2022-04-20T17:30:00.000000000
2022-09-29T07:30:00.000000000 2022-10-03T14:00:00.000000000
2021-10-08T19:00:00.000000000 2021-10-10T08:00:00.000000000
2022-12-05T00:00:00.000000000 2022-12-08T12:30:00.000000000
2023-04-04T00:30:00.000000000 2023-04-05T16:30:00.000000000
2023-05-22T15:00:00.000000000 2023-05-22T17:30:00.000000000
2022-02-21T12:00:00.000000000 2022-02-24T08:30:00.000000000
2021-10-23T18:30:00.000000000 2021-10-24T07:30:00.000000000
2022-04-07T21:30:00.000000000 2022-04-11T00:00:00.000000000


# Figure for blowing snow event

In [8]:
DATA_DIR = '/storage/dlhogan/precipitation-rodeo/data/for_analysis'
# Define your root and sites
DATA_DIR = Path(DATA_DIR)
STORAGE_DIR = '/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo/04_products/figures/events'
SRC = "asfs" # one of [bb, sail, asfs, sos, '']
PRODUCT = "events" # or events
WITH_MET = True  # or True
RAW_OR_NORMALIZED = "raw"  # or raw
file_dest = get_file_destination(DATA_DIR, SRC, PRODUCT, WITH_MET, RAW_OR_NORMALIZED)
print(f"Loading data from: {file_dest}")
ds = xr.open_dataset(file_dest)

Loading data from: /storage/dlhogan/precipitation-rodeo/data/for_analysis/kettle_ponds/events_with_raw_met/kettle_ponds_precipitation_event_comparisons_asfs_with_raw_met.nc


In [9]:
# set benchmark site 
SITE = 'kettle_ponds'  # one of [gothic, kettle_ponds]
BENCHMARK = 'billy_barr_precip'
EVENT = 1

INSTRUMENT = "splash_ld_unadjusted"
OUTPUT_DIR = os.path.join(STORAGE_DIR, f"{SITE}/{INSTRUMENT}")

ppt_ds_kp = kettle_ponds_ppt_ds
ppt_ds_gt = gothic_ppt_ds
met_ds = sos_ds
# if output dir does not exist
os.makedirs(OUTPUT_DIR+f'/bb_met', exist_ok=True)


In [23]:
# calculate % of bias during blowing snow event
blowing_snow_mask = ((met_ds['SF_avg_1m_ue'] + met_ds['SF_avg_2m_ue']) > 0.01).sel(time=slice('2022-12-01', '2023-03-31'))
ppt_kp_blowing_snow = (abs(ppt_ds_kp - ppt_ds_gt[BENCHMARK])).where(blowing_snow_mask, drop=True).sum()
ppt_gt_blowing_snow = (abs(ppt_ds_gt - ppt_ds_gt[BENCHMARK])).where(blowing_snow_mask, drop=True).sum()
total_bias_ld_kp = (abs(ppt_ds_kp - ppt_ds_gt[BENCHMARK])).sel(time=slice('2022-12-01', '2023-03-31')).sum()
total_bias_ld_gt = (abs(ppt_ds_gt - ppt_ds_gt[BENCHMARK])).sel(time=slice('2022-12-01', '2023-03-31')).sum()
bias_during_blowing_snow_kp = ((ppt_kp_blowing_snow)/ total_bias_ld_kp) * 100
bias_during_blowing_snow_gt = ((ppt_gt_blowing_snow)/ total_bias_ld_gt) * 100
print(f"Bias during blowing snow event for KP: {bias_during_blowing_snow_kp['splash_ld_unadjusted']:.2f}%")
print(f"Bias during blowing snow event for GT: {bias_during_blowing_snow_gt['sail_ld_uncorrected']:.2f}%")
print(f"Bias during blowing snow event for GT: {bias_during_blowing_snow_gt['sail_org']:.2f}%")
print(f"Bias during blowing snow event for GT: {bias_during_blowing_snow_gt['sail_pwd']:.2f}%")
# percent of time blowing snow conditions occur
blowing_snow_duration = blowing_snow_mask.sum().values
total_duration = met_ds['time'].size
blowing_snow_percentage = (blowing_snow_duration / total_duration) * 100
print(f"Blowing snow conditions occur {blowing_snow_percentage:.2f}% of time")

Bias during blowing snow event for KP: 34.91%
Bias during blowing snow event for GT: 29.36%
Bias during blowing snow event for GT: 29.70%
Bias during blowing snow event for GT: 23.79%
Blowing snow conditions occur 9.56% of time


In [10]:
import datetime
event_ds = ds.sel(event_id=EVENT, test_instrument=INSTRUMENT, benchmark=BENCHMARK)

start, end = pd.to_datetime(event_ds['start_time'].values), pd.to_datetime(event_ds['end_time'].values) + datetime.timedelta(hours=6)
event_ppt_kp = ppt_ds_kp.sel(time=slice(start, end))
event_ppt_gt = ppt_ds_gt.sel(time=slice(start, end))
event_met = met_ds.sel(time=slice(start, end))
asfs_event_met = kettle_ponds_met_ds.sel(time=slice(start, end))
event_bb_met = billy_met_ds.sel(time=slice(start, end))

# calcualte wind direction
u = event_met['u'].metpy.convert_units('m/s')
v = event_met['v'].metpy.convert_units('m/s')
wind_dir = calc.wind_direction(u, v).metpy.dequantify()
event_met['wind_dir'] = wind_dir
event_ppt_rate_kp = (event_ppt_kp*2)#.rolling(time=2, center=True).mean()  # mm per 30 min to mm per hour
event_ppt_rate_gt = (event_ppt_gt*2)#.rolling(time=2, center=True).mean()  # mm per 30 min to mm per hour
print(event_ds['start_time'].values, event_ds['end_time'].values)

# fill values after 2022-12-22T09:00:00.000000000 with 0
event_ppt_kp['splash_pluvio'] = event_ppt_kp['splash_pluvio'].where(event_ppt_kp['time'] <= np.datetime64('2022-12-22T06:03:00.000000000'), 0)

2022-12-20T19:00:00.000000000 2022-12-22T09:30:00.000000000


In [12]:
# make a plotly plot of the event with ppt and met data
fig = make_subplots(rows=3, cols=1, 
                    shared_xaxes=True, 
                    subplot_titles=("Precipitation", "10-m Wind Speed", "Blowing Snow Flux"),
                    vertical_spacing=0.05,
                    specs=[[{"secondary_y": True}],  # only first subplot
                            [{"secondary_y": False}],
                            [{}]])
# update size of subplot titles
for i in range(1, 4):
    fig.layout.annotations[i-1].update(font=dict(size=20))
#### Plot 1: Precipitation ####
fig.add_trace(go.Scatter(x=event_ppt_kp['time'].values, 
                        y=event_ppt_kp[INSTRUMENT].cumsum().values, 
                        name=f'KP LD unadjusted:{event_ppt_kp[INSTRUMENT].sum().values:.1f} mm',
                        mode='lines',
                        line=dict(width=4, color=colors[1])), 
                        row=1, col=1, )
fig.add_trace(go.Scatter(x=event_ppt_gt['time'].values, 
                        y=event_ppt_gt["sail_ld_uncorrected"].cumsum().values, 
                        name=f'GT LD unadjusted: {event_ppt_gt["sail_ld_uncorrected"].sum().values:.1f} mm',
                        mode='lines',
                        line=dict(width=4, color=colors[2])), 
                        row=1, col=1, )
fig.add_trace(go.Scatter(x=event_ppt_kp['time'].values, 
                    y=event_ppt_kp[BENCHMARK].cumsum().values, 
                    name=f'Reference:{event_ppt_kp[BENCHMARK].sum().values:.1f} mm',
                    mode='lines',
                    line=dict(width=4, color=colors[0])), 
                    row=1, col=1, )
fig.add_trace(go.Scatter(x=event_ppt_kp['time'].values, 
                        y=event_ppt_kp["splash_pluvio"].cumsum().values, 
                        name=f'KP weighing bucket:{event_ppt_kp["splash_pluvio"].sum().values:.1f} mm',
                        mode='lines',
                        line=dict(width=4, color=colors[3])), 
                        row=1, col=1, )
fig.add_trace(go.Scatter(x=event_ppt_gt['time'].values, 
                        y=event_ppt_gt["sail_pluvio"].cumsum().values, 
                        name=f'GT weighing bucket:{event_ppt_gt["sail_pluvio"].sum().values:.1f} mm',
                        mode='lines',
                        line=dict(width=2, color=colors[-1])), 
                        row=1, col=1, )
# Precipitation rate
# secondary y-axis for precipitation rate
fig.add_trace(go.Scatter(x=event_ppt_rate_kp['time'].values, 
                        y=event_ppt_rate_kp[BENCHMARK].values, 
                        name=f'Reference Rate (mm/hr)',
                        showlegend=False,
                        mode='lines',
                        line=dict(width=2, dash='dot', color=colors[0]),
                        yaxis='y2'),
                        secondary_y=True,
                        row=1, col=1, )
fig.add_trace(go.Scatter(x=event_ppt_rate_kp['time'].values, 
                        y=event_ppt_rate_kp[INSTRUMENT].values, 
                        name=f'SPLASH LD Unadjusted Rate (mm/hr)',
                        mode='lines',
                        showlegend=False,
                        line=dict(width=2, dash='dot', color=colors[1]),
                        yaxis='y2'),
                        secondary_y=True,
                        row=1, col=1, )
fig.add_trace(go.Scatter(x=event_ppt_rate_gt['time'].values, 
                        y=event_ppt_rate_gt["sail_ld_uncorrected"].values, 
                        name='SAIL LD Unadjusted',
                        mode='lines',
                        showlegend=False,
                        line=dict(width=2, dash='dot', color=colors[2]),
                        yaxis='y2'),
                        secondary_y=True,
                        row=1, col=1, )
#### Plot 2: Wind Speed ####
fig.add_trace(go.Scatter(x=event_met['time'].values, 
                        y=event_met['wspd_vec_mean'].values, 
                        mode='lines', 
                        name='Wind Speed (m/s)',
                        showlegend=False,
                        line=dict(width=4, color='black')), 
                        row=2, col=1)
# Wind direction as secondary y-axis
# fig.add_trace(go.Scatter(x=event_met['time'].values, 
#                         y=event_met['wind_dir'].values, 
#                         mode='markers', 
#                         name='Wind Direction (°)',
#                         showlegend=False,
#                         line=dict(width=2, dash='dot', color='gray')),
#                         secondary_y=True,
#                         row=2, col=1)
#### Plot 5: Blowing Snow Flux ####
fig.add_trace(go.Scatter(x=event_met['time'].values, 
                        y=(event_met['SF_avg_1m_ue']+ event_met['SF_avg_2m_ue']).interpolate_na(method='linear', dim='time').values+0.001,
                        mode='lines', 
                        name='Blowing Snow Flux (g/m²/s)',
                        showlegend=False,
                        line=dict(width=4, color='black')),
                        row=3, col=1)
# make log scale
fig.update_yaxes(type="log", row=3, col=1)
# add minor ticks
fig.update_yaxes(dtick=1, row=3, col=1)
# specify ticks 
fig.update_yaxes(tickvals=[0, 0.01, 0.1, 1, 10, 100], row=4, col=1)

# fig.update_layout( title_text=f'Event ID: {event_ds["event_id"].values} from {event_ds["start_time"].dt.date.values} to {event_ds["end_time"].dt.date.values}')

# update yaxis titles
fig.update_yaxes(title_text="Cumulative<br>Precipitation (mm) [solid]", row=1, col=1)
fig.update_layout(yaxis2=dict(title='Precipitation Rate<br>(mm/hr) [dashed]', 
                              overlaying='y', side='right', position=0.15))
fig.update_yaxes(title_text="Wind Speed<br>(m/s)", row=2, col=1)
fig.update_yaxes(title_text="Blowing Snow<br>Flux (g/m²/s)", row=3, col=1)

# Update layout to enable vertical line (spike) on all subplots
for axis in ['xaxis', 'xaxis2', 'xaxis3']:
    fig.layout[axis].update(
        showspikes=True,
        spikemode='across',
        spikesnap='cursor',
        spikecolor='gray',
        spikethickness=1
    )
# update tick size
fig.update_layout(font=dict(size=16))
# move the legend to the bottom with 4 columns
fig.update_layout(legend=dict(orientation="v", 
                              yanchor="top", 
                              y=0.99, 
                              xanchor="left", 
                              x=0.01, 
                              bgcolor='rgba(255,255,255,0.7)', # Fully transparent background
                              traceorder="normal", font=dict(size=14),
                              title="Event Totals:"
                              ), 
                height=800, width=700,
                hovermode='x',  # vertical line and combined tooltip
                )
fig.update_yaxes(showgrid=False, secondary_y=True, row=1, col=1)
fig.update_yaxes(showgrid=False, secondary_y=True, row=2, col=1)
fig.update_yaxes(showgrid=False, secondary_y=True, row=3, col=1)
# decrease vertical white space between each subplot
fig.update_layout(margin=dict(t=100, b=100, l=50, r=50), 
                title_font_size=24)
fig.update_xaxes(range=[pd.to_datetime(event_bb_met['time'].min().values), pd.to_datetime(event_bb_met['time'].max().values) ])

SAVE_DIR = '/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo/04_products/figures/for_manuscript'
width_pixels = 3 * 200  # 6 inches * 600 dpi
height_pixels = 4 * 200  # 8 inches * 600 dpi
fig.write_image(f'{SAVE_DIR}/blowing_snow_event.png', scale=2,
                height=height_pixels, width=width_pixels)
fig.show()

# Figure for power outage event

In [29]:
DATA_DIR = '/storage/dlhogan/precipitation-rodeo/data/for_analysis'
# Define your root and sites
DATA_DIR = Path(DATA_DIR)
STORAGE_DIR = '/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo/04_products/figures/events'
SRC = "sail" # one of [bb, sail, asfs, sos, '']
PRODUCT = "events" # or events
WITH_MET = True  # or True
RAW_OR_NORMALIZED = "raw"  # or raw
file_dest = get_file_destination(DATA_DIR, SRC, PRODUCT, WITH_MET, RAW_OR_NORMALIZED)
print(f"Loading data from: {file_dest}")
ds = xr.open_dataset(file_dest)

Loading data from: /storage/dlhogan/precipitation-rodeo/data/for_analysis/gothic/events_with_raw_met/gothic_precipitation_event_comparisons_sail_with_raw_met.nc


In [30]:
# set benchmark site 
SITE = 'gothic'  # one of [gothic, kettle_ponds]
BENCHMARK = 'billy_barr_precip'
EVENT = 1

INSTRUMENT = "sail_pluvio"
OUTPUT_DIR = os.path.join(STORAGE_DIR, f"{SITE}/{INSTRUMENT}")

ppt_ds_kp = kettle_ponds_ppt_ds
ppt_ds_gt = gothic_ppt_ds
met_ds = sos_ds
# if output dir does not exist
os.makedirs(OUTPUT_DIR+f'/bb_met', exist_ok=True)

In [31]:
# how often did outages occur?
gt_outage_ds = xr.open_dataset("/storage/dlhogan/precipitation-rodeo/data/processed/SAIL/pluvio_30min.nc")
gt_missing_data = gt_outage_ds['accum_rtnrt'].sel(time=slice(ppt_ds_gt['time'].min(), ppt_ds_gt['time'].max())).isnull().sum()
gt_precip_flag = gt_outage_ds['accum_rtnrt'].isnull() & (ppt_ds_gt['billy_barr_precip'] > 0)

print("Percent of time an outage occurred during precipitation at GT: ", (gt_precip_flag.sum()/(ppt_ds_gt['billy_barr_precip'] > 0).sum()).values*100)
print("Percent of total time outages occurred: ", (gt_missing_data.values/len(gt_outage_ds['accum_rtnrt']))*100)
print("Percent of outages during precipitation relative to total number of outages: ", (gt_precip_flag.sum().values/gt_missing_data.values)*100)


Percent of time an outage occurred during precipitation at GT:  1.5300918055083303
Percent of total time outages occurred:  1.6602564102564104
Percent of outages during precipitation relative to total number of outages:  8.687258687258687


In [32]:
kp_outage_ds = xr.open_dataset("/storage/dlhogan/precipitation-rodeo/data/processed/SPLASH/SPLASH_kp_laser_disdrometer_30min.nc")

# how often did outages occur?
kp_missing_data = kp_outage_ds['qc_missing_precip'].sel(time=slice(ppt_ds_kp['time'].min(), ppt_ds_kp['time'].max())).sum()
kp_precip_flag = kp_outage_ds['qc_missing_precip'] & (ppt_ds_kp['billy_barr_precip'] > 0)

print("Percent of time an outage occurred during precipitation at kp: ", (kp_precip_flag.sum()/(ppt_ds_kp['billy_barr_precip'] > 0).sum()).values*100)
print("Percent of total time outages occurred: ", (kp_missing_data.values/len(kp_outage_ds['qc_missing_precip']))*100)
print("Percent of outages during precipitation relative to total number of outages: ", (kp_precip_flag.sum().values/kp_missing_data.values)*100)


Percent of time an outage occurred during precipitation at kp:  4.556273376402585
Percent of total time outages occurred:  4.6980262380333295
Percent of outages during precipitation relative to total number of outages:  8.427672955974842


In [33]:
event_ds = ds.sel(event_id=EVENT, test_instrument=INSTRUMENT, benchmark=BENCHMARK)

start, end = pd.to_datetime(event_ds['start_time'].values), pd.to_datetime(event_ds['end_time'].values) + np.timedelta64(3, 'h')
event_ppt_kp = ppt_ds_kp.sel(time=slice(start, end))
event_ppt_gt = ppt_ds_gt.sel(time=slice(start, end))
event_met = met_ds.sel(time=slice(start, end))
asfs_event_met = kettle_ponds_met_ds.sel(time=slice(start, end))
event_bb_met = billy_met_ds.sel(time=slice(start, end))

# calcualte wind direction
u = event_met['u'].metpy.convert_units('m/s')
v = event_met['v'].metpy.convert_units('m/s')
wind_dir = calc.wind_direction(u, v).metpy.dequantify()
event_met['wind_dir'] = wind_dir
event_ppt_rate_kp = (event_ppt_kp*2)#.rolling(time=2, center=True).mean()  # mm per 30 min to mm per hour
event_ppt_rate_gt = (event_ppt_gt*2)#.rolling(time=2, center=True).mean()  # mm per 30 min to mm per hour
print(event_ds['start_time'].values, event_ds['end_time'].values)

2023-03-10T05:00:00.000000000 2023-03-13T04:30:00.000000000


In [34]:
# fill missing data with event_bb_met data where possible
for var in ['temp_mean', 'atmos_pressure', 'wspd_vec_mean', 'wind_dir']:
    event_met[var] = event_met[var].combine_first(event_bb_met[var])

In [36]:
# make a plotly plot of the event with ppt and met data
fig = make_subplots(rows=1, cols=1, 
                    subplot_titles=("",),
                    # vertical_spacing=0.05,
                    specs=[[{"secondary_y": True}]],  # only first subplot
                            )
# update size of subplot titles
# fig.layout.annotations[0].update(font=dict(size=24))
#### Plot 1: Precipitation ####
fig.add_trace(go.Scatter(x=event_ppt_kp['time'].values, 
                    y=event_ppt_kp[BENCHMARK].cumsum().values, 
                    name=f'Reference: {event_ppt_kp[BENCHMARK].sum().values:.1f} mm',
                    mode='lines',
                    line=dict(width=4, color=colors[0])), 
                    row=1, col=1, )
fig.add_trace(go.Scatter(x=event_ppt_gt['time'].values, 
                        y=event_ppt_gt[INSTRUMENT].cumsum().values, 
                        name=f'Gothic WB: {event_ppt_gt[INSTRUMENT].sum().values:.1f} mm',
                        mode='lines',
                        line=dict(width=4, color=colors[1])), 
                        row=1, col=1, )
# Precipitation rate
# secondary y-axis for precipitation rate
fig.add_trace(go.Scatter(x=event_ppt_rate_gt['time'].values, 
                        y=event_ppt_rate_gt[BENCHMARK].values, 
                        name=f'Reference',
                        showlegend=False,
                        mode='lines',
                        line=dict(width=2, dash='dot', color=colors[0]),
                        yaxis='y2'),
                        secondary_y=True,
                        row=1, col=1, )
fig.add_trace(go.Scatter(x=event_ppt_rate_gt['time'].values, 
                        y=event_ppt_rate_gt[INSTRUMENT].values, 
                        name='SAIL WB',
                        mode='lines',
                        showlegend=False,
                        line=dict(width=2, dash='dot', color=colors[1]),
                        yaxis='y2'),
                        secondary_y=True,
                        row=1, col=1, )

# fig.update_layout( title_text=f'Event ID: {event_ds["event_id"].values} from {event_ds["start_time"].dt.date.values} to {event_ds["end_time"].dt.date.values}')
# shade the area of the outage between march 10 23:00 and march 11 15:30
fig.add_vrect(x0="2023-03-10 23:00", x1="2023-03-11 15:30", 
              fillcolor="red", opacity=0.2, line_width=0, row=1, col=1)
# add annotation for the outage
fig.add_annotation(x="2023-03-11 07:30", y=event_ppt_kp[BENCHMARK].cumsum().max().values*1.8,
                   text="Outage Period", showarrow=False, font=dict(size=16, color='black', weight='bold'),
                     row=1, col=1)
# update yaxis titles
fig.update_yaxes(title_text="Cumulative Precipitation<br>(mm) [solid]", row=1, col=1)
fig.update_layout(yaxis2=dict(title='Precipitation Rate<br>(mm/hr) [dashed]', overlaying='y', side='right', position=0.15))

# Update layout to enable vertical line (spike) on all subplots
fig.layout['xaxis'].update(
    showspikes=True,
    spikemode='across',
    spikesnap='cursor',
    spikecolor='gray',
    spikethickness=1)
    
# update tick size
fig.update_layout(font=dict(size=16))
# move the legend to the bottom with 4 columns
fig.update_layout(legend=dict(orientation="v", 
                              yanchor="top", 
                              y=0.99, 
                              xanchor="left", 
                              x=0.51, 
                              bgcolor='rgba(255,255,255,0.7)', # Fully transparent background
                              traceorder="normal", font=dict(size=14),
                              title="Event Totals:",
                              ), 
                height=400, width=700,
                hovermode='x',  # vertical line and combined tooltip
                )
fig.update_yaxes(showgrid=False, secondary_y=True, row=1, col=1)
# update y range on first subplot first axis
fig.update_yaxes(range=[0, event_ppt_kp[BENCHMARK].cumsum().max().values*2], secondary_y=False,row=1, col=1)
# decrease vertical white space between each subplot
fig.update_layout(margin=dict(t=100, b=100, l=50, r=50), 
                title_font_size=24)
fig.update_xaxes(range=[pd.to_datetime(event_met['time'].min().values), pd.to_datetime(event_met['time'].max().values) ])

# rotate x-axis labels
fig.update_xaxes(tickangle=45)

SAVE_DIR = '/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo/04_products/figures/for_manuscript'
width_pixels = 3.1 * 200  # 6 inches * 600 dpi
height_pixels = 2.1 * 200  # 6 inches * 600 dpi
fig.write_image(f'{SAVE_DIR}/power_outage_event.png', scale=3,
                height=height_pixels, width=width_pixels)
fig.show()

# Figure for gauge burial

In [6]:
DATA_DIR = '/storage/dlhogan/precipitation-rodeo/data/for_analysis'
# Define your root and sites
DATA_DIR = Path(DATA_DIR)
STORAGE_DIR = '/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo/04_products/figures/events'
SRC = "asfs" # one of [bb, sail, asfs, sos, '']
PRODUCT = "events" # or events
WITH_MET = True  # or True
RAW_OR_NORMALIZED = "raw"  # or raw
file_dest = get_file_destination(DATA_DIR, SRC, PRODUCT, WITH_MET, RAW_OR_NORMALIZED)
print(f"Loading data from: {file_dest}")
ds = xr.open_dataset(file_dest)

Loading data from: /storage/dlhogan/precipitation-rodeo/data/for_analysis/kettle_ponds/events_with_raw_met/kettle_ponds_precipitation_event_comparisons_asfs_with_raw_met.nc


In [11]:
# set benchmark site 
SITE = 'kettle_ponds'  # one of [gothic, kettle_ponds]
BENCHMARK = 'billy_barr_precip'
EVENT = 1

INSTRUMENT = "splash_pluvio"
OUTPUT_DIR = os.path.join(STORAGE_DIR, f"{SITE}/{INSTRUMENT}")

ppt_ds_kp = kettle_ponds_ppt_ds
ppt_ds_gt = gothic_ppt_ds
met_ds = sos_ds
# if output dir does not exist
os.makedirs(OUTPUT_DIR+f'/bb_met', exist_ok=True)

In [12]:
event_ds = ds.sel(event_id=EVENT, test_instrument=INSTRUMENT, benchmark=BENCHMARK)

start, end = pd.to_datetime(event_ds['start_time'].values), pd.to_datetime(event_ds['end_time'].values)
event_ppt_kp = ppt_ds_kp.sel(time=slice(start, end))
event_ppt_gt = ppt_ds_gt.sel(time=slice(start, end))
event_met = met_ds.sel(time=slice(start, end))
asfs_event_met = kettle_ponds_met_ds.sel(time=slice(start, end))
event_bb_met = billy_met_ds.sel(time=slice(start, end))

# calcualte wind direction
u = event_met['u'].metpy.convert_units('m/s')
v = event_met['v'].metpy.convert_units('m/s')
wind_dir = calc.wind_direction(u, v).metpy.dequantify()
event_met['wind_dir'] = wind_dir
event_ppt_rate_kp = (event_ppt_kp*2)#.rolling(time=2, center=True).mean()  # mm per 30 min to mm per hour
event_ppt_rate_gt = (event_ppt_gt*2)#.rolling(time=2, center=True).mean()  # mm per 30 min to mm per hour
print(event_ds['start_time'].values, event_ds['end_time'].values)

2023-03-30T10:00:00.000000000 2023-04-01T17:30:00.000000000


In [17]:
# make a plotly plot of the event with ppt and met data
fig = make_subplots(rows=1, cols=1, 
                    shared_xaxes=True, 
                    subplot_titles=("Precipitation",),
                    vertical_spacing=0.1,
                    specs=[[{"secondary_y": True}],  # only first subplot
                            # [{"secondary_y": True}],
                            ])
# update size of subplot titles

fig.layout.annotations[0].update(font=dict(size=18))
#### Plot 1: Precipitation ####
fig.add_trace(go.Scatter(x=event_ppt_kp['time'].values, 
                    y=event_ppt_kp[BENCHMARK].cumsum().values, 
                    name=f'Benchmark: {event_ppt_kp[BENCHMARK].sum().values:.1f} mm',
                    mode='lines',
                    line=dict(width=4, color=colors[0])), 
                    row=1, col=1, )
fig.add_trace(go.Scatter(x=event_ppt_kp['time'].values, 
                        y=event_ppt_kp["splash_pluvio"].cumsum().values, 
                        name=f'SPLASH WB: {event_ppt_kp["splash_pluvio"].sum().values:.1f} mm',
                        mode='lines',
                        line=dict(width=4, color=colors[1])), 
                        row=1, col=1, )
# secondary y-axis for precipitation rate
fig.add_trace(go.Scatter(x=event_ppt_rate_gt['time'].values, 
                        y=event_ppt_rate_gt[BENCHMARK].values, 
                        name=f'Reference',
                        showlegend=False,
                        mode='lines',
                        line=dict(width=2, dash='dot', color=colors[0]),
                        yaxis='y2'),
                        secondary_y=True,
                        row=1, col=1, )
fig.add_trace(go.Scatter(x=event_ppt_rate_kp['time'].values, 
                        y=event_ppt_rate_kp["splash_pluvio"].values, 
                        name=f'SPLASH WB',
                        mode='lines',
                        showlegend=False,
                        line=dict(width=2, dash='dot', color=colors[1]),
                        yaxis='y2'),
                        secondary_y=True,
                        row=1, col=1, )
#### Plot 2: Wind Speed ####
# fig.add_trace(go.Scatter(x=event_met['time'].values, 
#                         y=event_met['wspd_vec_mean'].values, 
#                         mode='lines', 
#                         name='Wind Speed (m/s)',
#                         showlegend=False,
#                         line=dict(width=4, color='black')), 
#                         row=2, col=1)
# # Wind direction as secondary y-axis
# fig.add_trace(go.Scatter(x=event_met['time'].values, 
#                         y=event_met['wind_dir'].values, 
#                         mode='markers', 
#                         name='Wind Direction (°)',
#                         showlegend=False,
#                         line=dict(width=2, dash='dot', color='gray')),
#                         secondary_y=True,
#                         row=2, col=1)

# fig.update_layout( title_text=f'Event ID: {event_ds["event_id"].values} from {event_ds["start_time"].dt.date.values} to {event_ds["end_time"].dt.date.values}')

# update yaxis titles
fig.update_yaxes(title_text="Cumulative Precipitation<br>(mm) [solid]", row=1, col=1)
fig.update_layout(yaxis2=dict(title='Precipitation Rate<br>(mm/hr) [dashed]', overlaying='y', side='right', position=0.15))
# fig.update_yaxes(title_text="Wind Speed<br>(m/s) [solid]", row=2, col=1)
# fig.update_yaxes(title_text="Wind Direction<br>(°) [dots]", secondary_y=True, row=2, col=1)

# Update layout to enable vertical line (spike) on all subplots
for axis in ['xaxis', 'xaxis2']:
    fig.layout['xaxis'].update(
        showspikes=True,
        spikemode='across',
        spikesnap='cursor',
        spikecolor='gray',
        spikethickness=1
    )
# update tick size
fig.update_layout(font=dict(size=16))
# move the legend to the bottom with 4 columns
fig.update_layout(legend=dict(orientation="v", 
                              yanchor="top", 
                              y=0.99, 
                              xanchor="left", 
                              x=0.4, 
                              bgcolor='rgba(255,255,255,0.7)', # Fully transparent background
                              traceorder="normal", font=dict(size=15),
                              title="Event Totals:"
                              ), 
                height=500, width=800,
                hovermode='x',  # vertical line and combined tooltip
                )
fig.update_yaxes(showgrid=False, secondary_y=True, row=1, col=1)
# fig.update_yaxes(showgrid=False, secondary_y=True, row=2, col=1)
# fig.update_yaxes(showgrid=False, secondary_y=True, row=3, col=1)
# update y range on first subplot first axis
fig.update_yaxes(range=[0, event_ppt_kp[INSTRUMENT].cumsum().max().values*2], secondary_y=False,row=1, col=1)
# fig.update_yaxes(showgrid=False, secondary_y=True, row=4, col=1)
# decrease vertical white space between each subplot
fig.update_layout(margin=dict(t=100, b=100, l=50, r=50), 
                title_font_size=24)
fig.update_xaxes(range=[pd.to_datetime(event_met['time'].min().values), pd.to_datetime(event_met['time'].max().values) ])

SAVE_DIR = '/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo/04_products/figures/for_manuscript'
width_pixels = 3 * 300  # 6 inches * 600 dpi
height_pixels = 3 * 200  # 6 inches * 600 dpi
fig.write_image(f'{SAVE_DIR}/buried_gauge.png', scale=2,
                height=height_pixels, width=width_pixels)
fig.show()

In [37]:
event_met

<xarray.Dataset> Size: 0B
Dimensions:         (time: 0)
Coordinates:
  * time            (time) datetime64[ns] 0B 
Data variables:
    rh_mean         (time) float32 0B ...
    temp_mean       (time) float32 0B ...
    atmos_pressure  (time) float32 0B ...
    u               (time) float32 0B 
    v               (time) float32 0B 
    SF_avg_1m_ue    (time) float32 0B ...
    SF_avg_2m_ue    (time) float32 0B ...
    wspd_vec_mean   (time) float32 0B 
    wind_dir        (time) float32 0B 
Attributes:
    project:                   SOS
    history:                   Created: 2024-03-04 08:06:41 +0000\n
    NIDAS_version:             v1.2.3
    calibration_file_path:     /h/eol/isfs/isfs/projects/SOS/ISFS/cal_files/$...
    project_config:            /h/eol/isfs/isfs/projects/SOS/ISFS/config/sos....
    wind3d_horiz_coordinates:  geographic
    file_length_seconds:       86400
    wind3d_horiz_rotation:     1
    wind3d_tilt_correction:    1

# Figure for short intense precip

In [298]:
DATA_DIR = '/storage/dlhogan/precipitation-rodeo/data/for_analysis'
# Define your root and sites
DATA_DIR = Path(DATA_DIR)
STORAGE_DIR = '/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo/04_products/figures/events'
SRC = "sail" # one of [bb, sail, asfs, sos, '']
PRODUCT = "events" # or events
WITH_MET = True  # or True
RAW_OR_NORMALIZED = "raw"  # or raw
file_dest = get_file_destination(DATA_DIR, SRC, PRODUCT, WITH_MET, RAW_OR_NORMALIZED)
print(f"Loading data from: {file_dest}")
ds = xr.open_dataset(file_dest)

Loading data from: /storage/dlhogan/precipitation-rodeo/data/for_analysis/gothic/events_with_raw_met/gothic_precipitation_event_comparisons_sail_with_raw_met.nc


In [299]:
# set benchmark site 
SITE = 'gothic'  # one of [gothic, kettle_ponds]
BENCHMARK = 'billy_barr_precip'
EVENT = 1

INSTRUMENT = "sail_tbg"
OUTPUT_DIR = os.path.join(STORAGE_DIR, f"{SITE}/{INSTRUMENT}")

ppt_ds_kp = kettle_ponds_ppt_ds
ppt_ds_gt = gothic_ppt_ds
met_ds = gothic_met_ds
# if output dir does not exist
os.makedirs(OUTPUT_DIR+f'/bb_met', exist_ok=True)

In [300]:
event_ds = ds.sel(event_id=EVENT, test_instrument=INSTRUMENT, benchmark=BENCHMARK)

start, end = pd.to_datetime(event_ds['start_time'].values), pd.to_datetime(event_ds['end_time'].values) + np.timedelta64(3, 'h')
event_ppt_kp = ppt_ds_kp.sel(time=slice(start, end))
event_ppt_gt = ppt_ds_gt.sel(time=slice(start, end))
event_met = met_ds.sel(time=slice(start, end))
asfs_event_met = kettle_ponds_met_ds.sel(time=slice(start, end))
event_bb_met = billy_met_ds.sel(time=slice(start, end))

# calcualte wind direction
u = event_met['u'].metpy.convert_units('m/s')
v = event_met['v'].metpy.convert_units('m/s')
wind_dir = calc.wind_direction(u, v).metpy.dequantify()
event_met['wind_dir'] = wind_dir
event_ppt_rate_kp = (event_ppt_kp*2)#.rolling(time=2, center=True).mean()  # mm per 30 min to mm per hour
event_ppt_rate_gt = (event_ppt_gt*2)#.rolling(time=2, center=True).mean()  # mm per 30 min to mm per hour
print(event_ds['start_time'].values, event_ds['end_time'].values)

2021-12-23T05:30:00.000000000 2022-01-01T13:00:00.000000000


In [9]:
# fill missing data with event_bb_met data where possible
for var in ['temp_mean', 'atmos_pressure', 'wspd_vec_mean', 'wind_dir']:
    event_met[var] = event_met[var].combine_first(event_bb_met[var])

In [ ]:
# make a plotly plot of the event with ppt and met data
fig = make_subplots(rows=3, cols=1, 
                    shared_xaxes=True, 
                    subplot_titles=("Precipitation", "3-m Temperature (solid) & Pressure (dashed)", "10-m Wind Speed (solid) & Direction (dots)"),
                    vertical_spacing=0.05,
                    specs=[[{"secondary_y": True}],  # only first subplot
                            [{"secondary_y": True}],
                            [{"secondary_y": True}],
                            ])
# update size of subplot titles
for i in range(1, 4):
    fig.layout.annotations[i-1].update(font=dict(size=18))
#### Plot 1: Precipitation ####
fig.add_trace(go.Scatter(x=event_ppt_gt['time'].values, 
                    y=event_ppt_gt[BENCHMARK].cumsum().values, 
                    name=f'Benchmark: {event_ppt_gt[BENCHMARK].sum().values:.1f} mm',
                    mode='lines',
                    line=dict(width=4, color=colors[0])), 
                    row=1, col=1, )
fig.add_trace(go.Scatter(x=event_ppt_gt['time'].values, 
                        y=event_ppt_gt["sail_pluvio"].cumsum().values, 
                        name=f'GT WB: {event_ppt_gt["sail_pluvio"].sum().values:.1f} mm',
                        mode='lines',
                        line=dict(width=4, color=colors[2])), 
                        row=1, col=1, )
fig.add_trace(go.Scatter(x=event_ppt_gt['time'].values, 
                        y=event_ppt_gt[INSTRUMENT].cumsum().values, 
                        name=f'GT TBG: {event_ppt_gt[INSTRUMENT].sum().values:.1f} mm',
                        mode='lines',
                        line=dict(width=4, color=colors[1])), 
                        row=1, col=1, )
# Precipitation rate
# secondary y-axis for precipitation rate
fig.add_trace(go.Scatter(x=event_ppt_rate_gt['time'].values, 
                        y=event_ppt_rate_gt[BENCHMARK].values, 
                        name=f'Reference',
                        showlegend=False,
                        mode='lines',
                        line=dict(width=2, dash='dot', color=colors[0]),
                        yaxis='y2'),
                        secondary_y=True,
                        row=1, col=1, )
fig.add_trace(go.Scatter(x=event_ppt_rate_gt['time'].values, 
                        y=event_ppt_rate_gt["sail_pluvio"].values, 
                        name='GT WB',
                        mode='lines',
                        showlegend=False,
                        line=dict(width=2, dash='dot', color=colors[2]),
                        yaxis='y2'),
                        secondary_y=True,
                        row=1, col=1, )
fig.add_trace(go.Scatter(x=event_ppt_rate_gt['time'].values, 
                        y=event_ppt_rate_gt[INSTRUMENT].values, 
                        name=f'GT TBG',
                        mode='lines',
                        showlegend=False,
                        line=dict(width=2, dash='dot', color=colors[1]),
                        yaxis='y2'),
                        secondary_y=True,
                        row=1, col=1, )

#### Plot 2: Temperature & Pressure ####
fig.add_trace(go.Scatter(x=event_met['time'].values, 
                        y=event_met['temp_mean'].values, 
                        mode='lines', 
                        name='Air Temperature (°C)',
                        showlegend=False,
                        line=dict(width=4, color='black')),
                        row=2, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=2, row=2, col=1)
fig.add_trace(go.Scatter(x=event_met['time'].values, 
                        y=event_bb_met['atmos_pressure'].values, 
                        name='Pressure (hPa)',
                        mode='lines',
                        showlegend=False,
                        line=dict(width=2, dash='dash', color='black'),
                        yaxis='y2'),
                        secondary_y=True,
                        row=2, col=1, )
#### Plot 3: Wind Speed ####
fig.add_trace(go.Scatter(x=event_met['time'].values, 
                        y=event_met['wspd_vec_mean'].values, 
                        mode='lines', 
                        name='Wind Speed (m/s)',
                        showlegend=False,
                        line=dict(width=4, color='black')), 
                        row=3, col=1)
# Wind direction as secondary y-axis
fig.add_trace(go.Scatter(x=event_met['time'].values, 
                        y=event_met['wind_dir'].values, 
                        mode='markers', 
                        name='Wind Direction (°)',
                        showlegend=False,
                        line=dict(width=2, dash='dot', color='gray')),
                        secondary_y=True,
                        row=3, col=1)

# fig.update_layout( title_text=f'Event ID: {event_ds["event_id"].values} from {event_ds["start_time"].dt.date.values} to {event_ds["end_time"].dt.date.values}')

# update yaxis titles
fig.update_yaxes(title_text="Cumulative<br>Precipitation (mm)", row=1, col=1)
fig.update_layout(yaxis2=dict(title='Precipitation Rate<br>(mm/hr)', overlaying='y', side='right', position=0.15))
fig.update_yaxes(title_text="Temperature<br>(°C)", row=2, col=1)
fig.update_yaxes(title_text="Pressure<br>(hPa)", secondary_y=True, row=2, col=1)
fig.update_yaxes(title_text="Wind Speed<br>(m/s)", row=3, col=1)
fig.update_yaxes(title_text="Wind Direction<br>(°)", secondary_y=True, row=3, col=1)

# Update layout to enable vertical line (spike) on all subplots
for axis in ['xaxis', 'xaxis2', 'xaxis3']:
    fig.layout[axis].update(
        showspikes=True,
        spikemode='across',
        spikesnap='cursor',
        spikecolor='gray',
        spikethickness=1
    )
# update tick size
fig.update_layout(font=dict(size=16))
# move the legend to the bottom with 4 columns
fig.update_layout(legend=dict(orientation="v", 
                              yanchor="top", 
                              y=0.99, 
                              xanchor="left", 
                              x=0.01, 
                              bgcolor='rgba(255,255,255,0.7)', #  transparent background
                              traceorder="normal", font=dict(size=14),
                              ), 
                height=800, width=700,
                hovermode='x',  # vertical line and combined tooltip
                )
fig.update_yaxes(showgrid=False, secondary_y=True, row=1, col=1)
fig.update_yaxes(showgrid=False, secondary_y=True, row=2, col=1)
fig.update_yaxes(showgrid=False, secondary_y=True, row=3, col=1)
# update y range on first subplot first axis
# fig.update_yaxes(range=[0, event_ppt_gt[INSTRUMENT].cumsum().max().values*2], secondary_y=False,row=1, col=1)
fig.update_yaxes(range=[0, 10], secondary_y=True,row=1, col=1)
fig.update_yaxes(showgrid=False, secondary_y=True, row=4, col=1)
# decrease vertical white space between each subplot
fig.update_layout(margin=dict(t=100, b=100, l=50, r=50), 
                title_font_size=24)
fig.update_xaxes(range=[pd.to_datetime(event_met['time'].min().values), pd.to_datetime(event_met['time'].max().values) ])

SAVE_DIR = '/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo/04_products/figures/for_manuscript'
width_pixels = 3 * 200  # 6 inches * 600 dpi
height_pixels = 4 * 200  # 6 inches * 600 dpi
fig.write_image(f'{SAVE_DIR}/precipitation_intensity.png', scale=2,
                height=height_pixels, width=width_pixels)
fig.show()

# Gridded Events

In [ ]:
SRC = "sail" # one of [bb, sail, asfs, sos, '']
PRODUCT = "gridded" # or events
WITH_MET = True  # or True
RAW_OR_NORMALIZED = "raw"  # or raw
BENCHMARK = 'sail_pluvio'
file_dest = get_file_destination(DATA_DIR, SRC, PRODUCT, WITH_MET, RAW_OR_NORMALIZED)
print(f"Loading data from: {file_dest}")
ds = xr.open_dataset(file_dest)

INSTRUMENT = 'prism_ppt'
SITE = 'gothic'
OUTPUT_DIR = os.path.join(STORAGE_DIR, f"{SITE}/{INSTRUMENT}")
# if output dir does not exist
os.makedirs(OUTPUT_DIR+f'/bb_met', exist_ok=True)
    
met_ds = gothic_met_ds
ppt_ds = gothic_ppt_ds

Loading data from: /storage/dlhogan/precipitation-rodeo/data/for_analysis/gothic/gridded_events_with_raw_met/gothic_gridded_precipitation_event_comparisons_sail_with_raw_met.nc


In [ ]:
for event in range(1,11):
    event_ds = ds.sel(event_id=event, test_instrument=INSTRUMENT, benchmark=BENCHMARK)
    event_met = met_ds.sel(time=slice(pd.to_datetime(event_ds['start_time'].values), pd.to_datetime(event_ds['end_time'].values)))
    event_ppt = ppt_ds.sel(time=slice(pd.to_datetime(event_ds['start_time'].values), pd.to_datetime(event_ds['end_time'].values)))
    # calcualte wind direction
    u = event_met['u'].metpy.convert_units('m/s')
    v = event_met['v'].metpy.convert_units('m/s')
    wind_dir = calc.wind_direction(u, v).metpy.dequantify()
    event_met['wind_dir'] = wind_dir
    # make a plotly plot of the event with ppt and met data
    fig = make_subplots(rows=5, cols=1, 
                        shared_xaxes=True, 
                        subplot_titles=("Precipitation", "Air Temperature", "Wind Speed", "Relative Humidity", "Pressure"),
                        vertical_spacing=0.05,
                        specs=[[{"secondary_y": True}],  # only first subplot
                                [{}], 
                                [{"secondary_y": True}],
                                [{}],
                                [{}]])
    # Cumulative precipitation
    fig.add_trace(go.Scatter(x=event_ppt['time'].values, 
                            y=event_ppt[BENCHMARK].cumsum().values, 
                            name=f'{BENCHMARK.replace("_", " ")} (mm)',
                            mode='lines',
                            line=dict(width=4, color=colors[0])), 
                            row=1, col=1, )
    fig.add_trace(go.Scatter(x=event_ppt['time'].values, 
                            y=np.full((event_ppt['time'].size,), event_ds['others_total'].values), 
                            name=f'{INSTRUMENT.replace("_", " ")} (mm)',
                            mode='lines',
                            line=dict(width=4, color=colors[1])), 
                            row=1, col=1, )
    # Temperature
    fig.add_trace(go.Scatter(x=event_met['time'].values, 
                            y=event_met['temp_mean'].values, 
                            mode='lines', 
                            name='Air Temperature (°C)',
                            line=dict(width=4,)),
                            row=2, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=5, row=2, col=1)
    # Wind speed
    fig.add_trace(go.Scatter(x=event_met['time'].values, 
                            y=event_met['wspd_vec_mean'].values, 
                            mode='lines', 
                            name='Wind Speed (m/s)',
                            line=dict(width=4,)), 
                            row=3, col=1)
    # Wind direction as secondary y-axis
    fig.add_trace(go.Scatter(x=event_met['time'].values, 
                            y=event_met['wind_dir'].values, 
                            mode='markers', 
                            name='Wind Direction (°)',
                            line=dict(width=2, dash='dot', color='gray')),
                            secondary_y=True,
                            row=3, col=1)
    fig.add_hline(y=180, line_dash="dash", line_color="black", line_width=5, row=3, col=1, secondary_y=True)
    # Plot relative humidity on 4th subplot
    fig.add_trace(go.Scatter(x=event_met['time'].values, 
                            y=event_met['rh_mean'].values, 
                            mode='lines', 
                            name='Relative Humidity (%)',
                            line=dict(width=4, color=colors[2])),
                            row=4, col=1)
    # Plot pressure on 5th subplot
    fig.add_trace(go.Scatter(x=event_met['time'].values,
                                y=event_met['atmos_pressure'].values*10,
                                mode='lines',
                                name='Pressure (hPa)',
                                line=dict(width=4, color='magenta')),
                                row=5, col=1)

    fig.update_layout( title_text=f'Event ID: {event_ds["event_id"].values} from {event_ds["start_time"].dt.date.values} to {event_ds["end_time"].dt.date.values}')

    # update yaxis titles
    fig.update_yaxes(title_text="Cumulative<br>Precipitation (mm)", row=1, col=1)
    fig.update_yaxes(title_text="Wind Speed (m/s)", row=3, col=1)
    fig.update_yaxes(title_text="Wind Direction (°)", secondary_y=True, row=3, col=1)
    fig.update_yaxes(title_text="Air<br>Temperature (°C)", row=2, col=1)


    # Update layout to enable vertical line (spike) on all subplots
    for axis in ['xaxis', 'xaxis2', 'xaxis3']:
        fig.layout[axis].update(
            showspikes=True,
            spikemode='across',
            spikesnap='cursor',
            spikecolor='gray',
            spikethickness=1
        )
    # move the legend to the bottom with 4 columns
    fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5, traceorder="normal", font=dict(size=18)), 
                    height=1000, width=800,
                    hovermode='x',  # vertical line and combined tooltip
                    )
    fig.update_yaxes(showgrid=False, secondary_y=True, row=1, col=1)
    fig.update_yaxes(showgrid=False, secondary_y=True, row=3, col=1)
    # decrease vertical white space between each subplot
    fig.update_layout(margin=dict(t=100, b=100, l=50, r=50), 
                    title_font_size=24)
    fig.update_xaxes(range=[pd.to_datetime(event_met['time'].min().values), pd.to_datetime(event_met['time'].max().values)])
    fig.write_image(f'{OUTPUT_DIR}/{SITE}_event_{event}_exploration_{INSTRUMENT}_vs_{BENCHMARK}.png', scale=2,
                    height=1000, width=850)
    print(f'Saved figure for event {event}')

Saved figure for event 1
Saved figure for event 2
Saved figure for event 3
Saved figure for event 4
Saved figure for event 5
Saved figure for event 6
Saved figure for event 7
Saved figure for event 8
Saved figure for event 9
Saved figure for event 10


In [ ]:
met_ds = billy_met_ds
for event in range(1,11):    
    event_ds = ds.sel(event_id=event, test_instrument=INSTRUMENT, benchmark=BENCHMARK)
# make a plotly plot of the event with ppt and met data
    fig = make_subplots(rows=5, cols=1, 
                        shared_xaxes=True, 
                        subplot_titles=("Precipitation", "Air Temperature", "Wind Speed", "Relative Humidity"),
                        vertical_spacing=0.05,
                        specs=[[{"secondary_y": True}],  # only first subplot
                                [{}], 
                                [{"secondary_y": True}],
                                [{}],
                                [{}]])
    # Cumulative precipitation
    fig.add_trace(go.Scatter(x=event_ppt['time'].values, 
                            y=event_ppt[BENCHMARK].cumsum().values, 
                            name=f'{BENCHMARK.replace("_", " ")} (mm)',
                            mode='lines',
                            line=dict(width=4, color=colors[0])), 
                            row=1, col=1, )
    fig.add_trace(go.Scatter(x=event_ppt['time'].values, 
                            y=np.full((event_ppt['time'].size,), event_ds['others_total'].values), 
                            name=f'{INSTRUMENT.replace("_", " ")} (mm)',
                            mode='lines',
                            line=dict(width=4, color=colors[1])), 
                            row=1, col=1, )
    # Temperature
    fig.add_trace(go.Scatter(x=met_ds['time'].values, 
                            y=met_ds['avAirTemp'].values, 
                            mode='lines', 
                            name='Air Temperature (°C)',
                            line=dict(width=4,)),
                            row=2, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=5, row=2, col=1)
    # Wind speed
    fig.add_trace(go.Scatter(x=met_ds['time'].values, 
                            y=met_ds['windSpeed'].values, 
                            mode='lines', 
                            name='Wind Speed (m/s)',
                            line=dict(width=4,)), 
                            row=3, col=1)
    # Wind direction as secondary y-axis
    fig.add_trace(go.Scatter(x=met_ds['time'].values, 
                            y=met_ds['windDirec'].values, 
                            mode='markers', 
                            name='Wind Direction (°)',
                            line=dict(width=2, dash='dot', color='gray')),
                            secondary_y=True,
                            row=3, col=1)
    fig.add_hline(y=180, line_dash="dash", line_color="black", line_width=5, row=3, col=1, secondary_y=True)
    # Plot relative humidity on 4th subplot
    fig.add_trace(go.Scatter(x=met_ds['time'].values, 
                            y=met_ds['relHumidty'].values, 
                            mode='lines', 
                            name='Relative Humidity (%)',
                            line=dict(width=4, color=colors[2])),
                            row=4, col=1)
    # Plot barometric pressure on 5th subplot
    fig.add_trace(go.Scatter(x=met_ds['time'].values, 
                            y=met_ds['baromPress'].values, 
                            mode='lines', 
                            name='Pressure (hPa)',
                            line=dict(width=4, color='magenta')),
                            row=5, col=1)

    fig.update_layout( title_text=f'Event ID: {event_ds["event_id"].values} from {event_ds["start_time"].dt.date.values} to {event_ds["end_time"].dt.date.values}')
    fig.layout['yaxis3'].update(showgrid=False)
    # update yaxis titles
    fig.update_yaxes(title_text="Cumulative<br>Precipitation (mm)", row=1, col=1)
    fig.update_layout(yaxis2=dict(title='Precipitation Rate<br>(mm/hr)', overlaying='y', side='right', position=0.15))
    fig.update_yaxes(title_text="Wind Speed (m/s)", row=3, col=1)
    fig.update_yaxes(title_text="Wind Direction (°)", secondary_y=True, row=3, col=1)
    fig.update_yaxes(title_text="Air Temperature (°C)", row=2, col=1)


    # Update layout to enable vertical line (spike) on all subplots
    for axis in ['xaxis', 'xaxis2', 'xaxis3']:
        fig.layout[axis].update(
            showspikes=True,
            spikemode='across',
            spikesnap='cursor',
            spikecolor='gray',
            spikethickness=1
        )
    # move the legend to the bottom with 4 columns
    fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5, traceorder="normal", font=dict(size=18)), 
                    height=1000, width=800,
                    hovermode='x',  # vertical line and combined tooltip
                    )
    fig.update_yaxes(showgrid=False, secondary_y=True, row=1, col=1)
    fig.update_yaxes(showgrid=False, secondary_y=True, row=3, col=1)
    # decrease vertical white space between each subplot
    fig.update_layout(margin=dict(t=100, b=100, l=50, r=50), 
                    title_font_size=24)
    # update x-axis to limits of the event
    fig.update_xaxes(range=[pd.to_datetime(met_ds['time'].min().values), pd.to_datetime(met_ds['time'].max().values)])
    fig.write_image(f'{OUTPUT_DIR}/bb_met/{SITE}_event_{event_ds["event_id"].values}_exploration_{INSTRUMENT}_vs_{BENCHMARK}.png', scale=2,
    height=1000, width=1000)